# QUESTION 6 — QUELS 3 000 SPECTATEURS CIBLER PAR SOIRÉE ?

## Problématique

Si on ne peut cibler que 3 000 spectateurs par soirée
avec une offre, lesquels ? Une offre envoyée à quelqu'un
qui ne consomme jamais est une offre perdue.

## Objectifs

- Estimer la propension des spectateurs à consommer au bar.
- Classer les spectateurs selon leur score de consommation.
- Comparer plusieurs stratégies de ciblage.
- Limiter le nombre d'offres envoyées aux non-consommateurs.
- Proposer une stratégie adaptée à la contrainte des
  3 000 spectateurs par soirée.

## Lien avec la question 5

La question 5 a montré les limites du KNN pour la
prédiction individuelle de la consommation au bar.

L'analyse complémentaire a cependant révélé des différences
de consommation selon les zones de la salle.

La question 6 exploite ces résultats pour construire
et évaluer différentes stratégies de ciblage commercial.

## 1. Importation et chargement des données

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Chargement des données nettoyées
billets = pd.read_csv('../data/processed/billets_clean.csv')

evenements = pd.read_csv('../data/processed/evenements_clean.csv',parse_dates=['date'])

# Vérification des dimensions
print("Dimensions des billets :", billets.shape)
print("Dimensions des événements :", evenements.shape)

Dimensions des billets : (700, 15)
Dimensions des événements : (96, 9)


## 2. Préparation des données

Les billets dont la consommation au bar est inconnue
sont exclus. La variable cible et les variables explicatives
retenues dans la question 5 sont réutilisées.

Le nombre de billets vendus est également récupéré pour
simuler la contrainte de ciblage des 3 000 spectateurs,
mais ne sera pas utilisé comme variable prédictive.

In [3]:
# Exclusion des consommations inconnues
donnees_q6 = billets.dropna(subset=['conso_bar_eur']).copy()

# Création de la variable cible
donnees_q6['conso_bar'] = (donnees_q6['conso_bar_eur'] > 0).astype(int)

# Jointure avec les données événementielles
donnees_q6 = donnees_q6.merge(
    evenements[[
        'evenement_id', 'genre', 'jour_semaine',
        'capacite', 'notoriete_artiste', 'billets_vendus'
    ]],
    on='evenement_id',
    how='left',
    validate='many_to_one'
)

# Sélection des variables explicatives
variables_numeriques = [
    'prix_paye_eur', 'achat_jours_avant',
    'age', 'notoriete_artiste'
]

variables_categorielles = [
    'zone', 'genre', 'jour_semaine', 'capacite'
]

variables_booleennes = ['deja_venu', 'invitation']

variables_explicatives = (
    variables_numeriques
    + variables_categorielles
    + variables_booleennes
)

# Vérification des résultats
print("Billets exploitables :", len(donnees_q6))
print("Consommateurs :", donnees_q6['conso_bar'].sum())
print("Non-consommateurs :", (donnees_q6['conso_bar'] == 0).sum())
print("Nombre de variables explicatives :", len(variables_explicatives))
print("Billets sans fréquentation connue :", donnees_q6['billets_vendus'].isna().sum())

Billets exploitables : 686
Consommateurs : 614
Non-consommateurs : 72
Nombre de variables explicatives : 10
Billets sans fréquentation connue : 22


## 3. Analyse de la contrainte des 3 000 spectateurs

L'échantillon ne contient pas suffisamment de billets
pour sélectionner directement 3 000 spectateurs par soirée.

Un quota proportionnel au nombre de billets vendus est
donc calculé pour chaque événement, puis appliqué au nombre
de billets exploitables de l'échantillon.

Les événements dont le nombre de ventes est inconnu
sont exclus de cette évaluation.

Cette approche constitue une simulation rétrospective
et non un ciblage réel de 3 000 spectateurs.

In [4]:
# Conserver les billets dont le nombre de ventes est connu et positif
donnees_ciblage = donnees_q6[
    donnees_q6['billets_vendus'].notna()
    & (donnees_q6['billets_vendus'] > 0)
].copy()

# Regroupement des billets par événement
quotas_soirees = donnees_ciblage.groupby('evenement_id').agg(
    billets_vendus=('billets_vendus', 'first'),
    taille_echantillon=('billet_id', 'count')
)

# Proportion de spectateurs à cibler
quotas_soirees['proportion_ciblage'] = (
    3000 / quotas_soirees['billets_vendus']
).clip(upper=1)

# Calcul du quota dans l'échantillon (arrondi au plus proche)
quotas_soirees['quota_ciblage'] = np.floor(
    quotas_soirees['taille_echantillon']
    * quotas_soirees['proportion_ciblage'] + 0.5
).astype(int)

# Synthèse des résultats
print("Billets retenus :", len(donnees_ciblage))
print("Événements retenus :", len(quotas_soirees))
print("Nombre total de billets à cibler :", quotas_soirees['quota_ciblage'].sum())
print("Événements avec un quota nul :", (quotas_soirees['quota_ciblage'] == 0).sum())

display(quotas_soirees.head(10).round(3))

Billets retenus : 664
Événements retenus : 91
Nombre total de billets à cibler : 375
Événements avec un quota nul : 0


,billets_vendus,taille_echantillon,proportion_ciblage,quota_ciblage
evenement_id,,,,
EV300,4861.0,9,0.617,6
EV301,6285.0,5,0.477,2
EV302,3639.0,4,0.824,3
EV304,5009.0,10,0.599,6
EV305,5674.0,9,0.529,5
EV306,5068.0,5,0.592,3
EV307,5047.0,9,0.594,5
EV308,6833.0,7,0.439,3
EV309,4916.0,5,0.610,3


**Interprétation :**

L'évaluation porte sur 664 billets répartis sur 91 événements.
Les stratégies devront sélectionner 375 billets, soit 56,48 %
de l'échantillon retenu, en respectant les quotas par soirée.

## 4. Définition du protocole d'évaluation

Une validation croisée stratifiée et groupée par événement
en cinq plis est utilisée pour éviter le chevauchement
des soirées entre entraînement et validation.

Les mêmes plis et les mêmes quotas sont utilisés pour
comparer équitablement toutes les stratégies de ciblage.

In [5]:
from sklearn.model_selection import StratifiedGroupKFold

# Préparation des données pour les modèles
donnees_ciblage = donnees_ciblage.reset_index(drop=True)

X = donnees_ciblage[variables_explicatives]
y = donnees_ciblage['conso_bar']
groupes = donnees_ciblage['evenement_id']

# Validation croisée groupée
cv_groupes = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

plis_cv = list(cv_groupes.split(X, y, groups=groupes))

# Vérification des plis
verification_plis = []

for i, (train_idx, val_idx) in enumerate(plis_cv, start=1):

    groupes_train = set(groupes.iloc[train_idx])
    groupes_val = set(groupes.iloc[val_idx])

    verification_plis.append({
        'Pli': i,
        'Billets entraînement': len(train_idx),
        'Billets validation': len(val_idx),
        'Événements entraînement': len(groupes_train),
        'Événements validation': len(groupes_val),
        'Non-consommateurs validation': (y.iloc[val_idx] == 0).sum(),
        'Événements communs': len(groupes_train & groupes_val)
    })

display(pd.DataFrame(verification_plis))

,Pli,Billets entraînement,Billets validation,Événements entraînement,Événements validation,Non-consommateurs validation,Événements communs
0,1,533,131,74,17,14,0
1,2,532,132,73,18,14,0
2,3,530,134,73,18,15,0
3,4,530,134,72,19,14,0
4,5,531,133,72,19,14,0


**Interprétation :**

Les cinq plis présentent des effectifs équilibrés et
aucun événement commun entre entraînement et validation.

La classe minoritaire est représentée dans chaque pli.
Ce protocole permet de comparer les stratégies de ciblage
sur des événements non utilisés pour leur entraînement.

## 5. Ciblage aléatoire (Baseline)

Le ciblage aléatoire constitue la stratégie de référence.

Pour chaque événement, le nombre attendu de consommateurs
sélectionnés est estimé à partir du quota de ciblage
et du taux de consommation observé.

Cette approche permet d'obtenir une référence stable,
sans dépendre d'un tirage aléatoire particulier.

Les stratégies prédictives seront comparées à cette
baseline en respectant les mêmes quotas par soirée.

In [6]:
# Nombre de consommateurs observés par événement
baseline_aleatoire = quotas_soirees.copy()

baseline_aleatoire['nombre_consommateurs'] = (
    donnees_ciblage.groupby('evenement_id')['conso_bar'].sum()
)

# Taux de consommation par événement
baseline_aleatoire['taux_consommation'] = (
    baseline_aleatoire['nombre_consommateurs']
    / baseline_aleatoire['taille_echantillon']
)

# Nombre attendu de consommateurs ciblés aléatoirement
baseline_aleatoire['consommateurs_attendus'] = (
    baseline_aleatoire['quota_ciblage']
    * baseline_aleatoire['taux_consommation']
)

# Nombre attendu de non-consommateurs ciblés
baseline_aleatoire['non_consommateurs_attendus'] = (
    baseline_aleatoire['quota_ciblage']
    - baseline_aleatoire['consommateurs_attendus']
)

# Résultats globaux
print("Nombre total d'offres :", baseline_aleatoire['quota_ciblage'].sum())

print("Consommateurs ciblés attendus :",
      round(baseline_aleatoire['consommateurs_attendus'].sum(), 2))

print("Non-consommateurs ciblés attendus :",
      round(baseline_aleatoire['non_consommateurs_attendus'].sum(), 2))

display(baseline_aleatoire[[
    'quota_ciblage',
    'taux_consommation',
    'consommateurs_attendus',
    'non_consommateurs_attendus'
]].head(10).round(3))

Nombre total d'offres : 375
Consommateurs ciblés attendus : 336.86
Non-consommateurs ciblés attendus : 38.14


,quota_ciblage,taux_consommation,consommateurs_attendus,non_consommateurs_attendus
evenement_id,,,,
EV300,6,1.000,6.000,0.000
EV301,2,0.800,1.600,0.400
EV302,3,1.000,3.000,0.000
EV304,6,0.900,5.400,0.600
EV305,5,0.778,3.889,1.111
EV306,3,1.000,3.000,0.000
EV307,5,1.000,5.000,0.000
EV308,3,0.714,2.143,0.857
EV309,3,0.800,2.400,0.600


**Interprétation :**

Le ciblage aléatoire permet de sélectionner en moyenne
336,86 consommateurs et 38,14 non-consommateurs pour
375 offres distribuées.

## 6. Construction du scoring avec KNN

Deux modèles KNN sont étudiés :

- Un modèle utilisant les 10 variables explicatives.
- Un modèle utilisant uniquement la zone.

Une configuration avec K = 5, une distance de Manhattan
et une pondération uniforme est retenue afin de produire
des scores permettant de classer les spectateurs.

Les scores sont obtenus par validation croisée groupée,
sans entraîner les modèles sur les événements évalués.

In [7]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_predict

# Prétraitement des 10 variables explicatives
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), variables_numeriques),
    ('cat', OneHotEncoder(handle_unknown='ignore',
                          sparse_output=False), variables_categorielles),
    ('bool', 'passthrough', variables_booleennes)
])

# KNN avec les 10 variables
knn_complet = Pipeline([
    ('preprocessing', preprocessor),
    ('knn', KNeighborsClassifier(
        n_neighbors=5,
        metric='manhattan',
        weights='uniform'
    ))
])

# KNN utilisant uniquement la zone
knn_zone = Pipeline([
    ('encodage', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    )),
    ('knn', KNeighborsClassifier(
        n_neighbors=5,
        metric='manhattan',
        weights='uniform'
    ))
])

# Estimation des scores hors pli
scores_ciblage = pd.DataFrame(index=X.index)

scores_ciblage['KNN complet'] = cross_val_predict(
    knn_complet, X, y,
    cv=plis_cv,
    method='predict_proba'
)[:, 1]

scores_ciblage['KNN zone'] = cross_val_predict(
    knn_zone, X[['zone']], y,
    cv=plis_cv,
    method='predict_proba'
)[:, 1]

# Vérification des scores obtenus
display(scores_ciblage.head(10))
print("Nombre de scores manquants :")
print(scores_ciblage.isna().sum())

,KNN complet,KNN zone
0,1.0,1.0
1,0.8,1.0
2,1.0,1.0
3,0.8,1.0
4,1.0,1.0
5,1.0,0.8
6,1.0,1.0
7,1.0,1.0
8,0.8,1.0
9,1.0,0.8


Nombre de scores manquants :
KNN complet    0
KNN zone       0
dtype: int64


## 7. Estimation directe des scores par zone

Cette stratégie attribue à chaque spectateur un score
correspondant au taux de consommation de sa zone.

Pour éviter les fuites de données, les taux sont calculés
uniquement sur les données d'entraînement de chaque pli,
puis appliqués aux spectateurs du pli de validation.

Cette méthode constitue une alternative simple et
interprétable aux modèles KNN.

In [8]:
# Initialisation des scores
scores_ciblage['Taux par zone'] = np.nan

# Estimation des taux par validation croisée groupée
for train_idx, val_idx in plis_cv:

    X_apprentissage = X.iloc[train_idx]
    y_apprentissage = y.iloc[train_idx]

    X_validation = X.iloc[val_idx]

    # Taux de consommation par zone sur l'entraînement
    taux_zone = y_apprentissage.groupby(
        X_apprentissage['zone']
    ).mean()

    # Attribution des scores aux spectateurs de validation
    scores_validation = X_validation['zone'].map(
        taux_zone
    ).fillna(y_apprentissage.mean())

    scores_ciblage.loc[
        X_validation.index, 'Taux par zone'
    ] = scores_validation

# Vérification des scores des trois stratégies
display(scores_ciblage.head(10).round(4))

print("Nombre de scores manquants :")
print(scores_ciblage.isna().sum())

print("\nNombre de scores distincts :")
print(scores_ciblage.nunique())

,KNN complet,KNN zone,Taux par zone
0,1.0,1.0,1.0000
1,0.8,1.0,0.9128
2,1.0,1.0,0.8824
3,0.8,1.0,0.9056
4,1.0,1.0,0.9128
5,1.0,0.8,0.9114
6,1.0,1.0,0.8364
7,1.0,1.0,0.8364
8,0.8,1.0,0.9048
9,1.0,0.8,0.8913


Nombre de scores manquants :
KNN complet      0
KNN zone         0
Taux par zone    0
dtype: int64

Nombre de scores distincts :
KNN complet       4
KNN zone          2
Taux par zone    16
dtype: int64


**Interprétation :**

Les trois stratégies produisent des scores sans valeurs
manquantes.

Le KNN complet présente 4 scores distincts, contre 2
pour le KNN zone et 16 pour l'estimation directe par zone.

Les nombreuses égalités de scores nécessitent une règle
de départage pour respecter les quotas de ciblage.

La diversité des scores ne garantit toutefois pas une
meilleure performance de ciblage.